# Step 2: Feature Engineering - Spatial Homologation
In this step, we use geospatial mathematics (cKDTree and 3D spherical projection) to match every Emergency Room center to its geographically closest environmental monitoring station.

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import math

# Project Rule 3: Text output must be generated for all visuals/mappings
print("=== SPATIAL HOMOLOGATION: Hospitals to SINCA Stations ===")

# 1. Load Hospitals
df_hosp = pd.read_csv('../data/raw/Atenciones de Urgencia/Establecimientos de Salud/establecimientos_20260714.csv', sep=';', encoding='latin1')
df_hosp = df_hosp.dropna(subset=['Latitud', 'Longitud'])
# Clean coordinates
df_hosp['Latitud'] = df_hosp['Latitud'].astype(str).str.replace(',', '.').astype(float)
df_hosp['Longitud'] = df_hosp['Longitud'].astype(str).str.replace(',', '.').astype(float)

hospitals_coords = df_hosp[['EstablecimientoCodigoAntiguo', 'EstablecimientoGlosa', 'Latitud', 'Longitud']].drop_duplicates(subset=['EstablecimientoCodigoAntiguo']).copy()

# 2. Load SINCA Stations
sinca_csv_path = '../data/raw/Registros de Calidad del Aire/Reportes SINCA/Data/DatosEstacioneSINCA.csv'
df_sinca = pd.read_csv(sinca_csv_path, sep=';', decimal='.', encoding='utf-8', skiprows=1)
df_sinca = df_sinca.dropna(subset=['latitud', 'longitud'])

sinca_coords = df_sinca[['estacion', 'region', 'latitud', 'longitud']].drop_duplicates(subset=['estacion']).reset_index(drop=True).copy()

print(f"Loaded {len(hospitals_coords)} Hospitals with valid coordinates.")
print(f"Loaded {len(sinca_coords)} SINCA Stations with valid coordinates.")

# 3. Convert Lat/Lon to 3D Cartesian coordinates for accurate spherical distance using cKDTree
R = 6371.0 # Earth radius in kilometers
sinca_coords['x'] = R * np.cos(np.radians(sinca_coords['latitud'])) * np.cos(np.radians(sinca_coords['longitud']))
sinca_coords['y'] = R * np.cos(np.radians(sinca_coords['latitud'])) * np.sin(np.radians(sinca_coords['longitud']))
sinca_coords['z'] = R * np.sin(np.radians(sinca_coords['latitud']))

hospitals_coords['x'] = R * np.cos(np.radians(hospitals_coords['Latitud'])) * np.cos(np.radians(hospitals_coords['Longitud']))
hospitals_coords['y'] = R * np.cos(np.radians(hospitals_coords['Latitud'])) * np.sin(np.radians(hospitals_coords['Longitud']))
hospitals_coords['z'] = R * np.sin(np.radians(hospitals_coords['Latitud']))

# Build the KDTree
tree = cKDTree(sinca_coords[['x', 'y', 'z']])

# Query the tree
distances, indices = tree.query(hospitals_coords[['x', 'y', 'z']])

hospitals_coords['nearest_sinca_station'] = sinca_coords.loc[indices, 'estacion'].values
hospitals_coords['nearest_sinca_region'] = sinca_coords.loc[indices, 'region'].values
# The Euclidean distance in 3D approximates the great-circle arc length closely for small distances
hospitals_coords['distance_to_sinca_km'] = distances 

# Project Rule 3 Text Output
print("\n--- Textual Data Output for Spatial Homologation (Top 15 mappings) ---")
display_df = hospitals_coords[['EstablecimientoGlosa', 'nearest_sinca_station', 'distance_to_sinca_km']].head(15)
print(display_df.to_string(index=False))
print("----------------------------------------------------------------------")

# Save the mapping
out_path = '../data/processed/hospital_sinca_mapping.csv'
hospitals_coords.to_csv(out_path, index=False)
print(f"\nMapping successfully saved to {out_path}")

=== SPATIAL HOMOLOGATION: Hospitals to SINCA Stations ===


Loaded 4025 Hospitals with valid coordinates.
Loaded 199 SINCA Stations with valid coordinates.

--- Textual Data Output for Spatial Homologation (Top 15 mappings) ---
                                           EstablecimientoGlosa nearest_sinca_station  distance_to_sinca_km
                                 Servicio MÃ©dico Legal Iquique         Alto Hospicio              8.688653
                                   Laboratorio ClÃ­nico Austral            Trapén Sur            120.641117
                                                   SUR Dalcahue            Trapén Sur            106.440711
                                  Posta de Salud Rural Cascadas          Puerto Varas             39.267417
                     Centro de RehabilitaciÃ³n de MinusvÃ¡lidos                Osorno             36.964984
                                   ClÃ­nica del Trabajador AChS              La Union              1.234795
                                         SAPU Belarmina Paredes     CESFAM L